# Day 73 — Re-ranking & hybrid search

Vectors capture meaning; keywords capture exact terms (names, IDs). **Hybrid** search
blends both, then **re-ranks**. > ✅ Offline.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
import hashlib, math

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

def keyword_score(query, text):
    return len(set(query.lower().split()) & set(text.lower().split()))

DOCS = {"d1": "error code 503 service unavailable", "d2": "the cat sat on the mat",
        "d3": "service mesh routing and load balancing"}

def hybrid(query, alpha=0.5):
    qe = cheap_embed(query)
    ranked = []
    for id, text in DOCS.items():
        vec = alpha * cosine(qe, cheap_embed(text))
        kw = (1 - alpha) * keyword_score(query, text)
        # TODO: append (vec + kw, id) to ranked
        raise NotImplementedError
    return sorted(ranked, reverse=True)

# print(hybrid("503 service"))

## 🔒 Solution

In [ ]:
import hashlib, math

def cheap_embed(text, dim=64):
    vec = [0.0] * dim
    for tok in text.lower().split():
        vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

def keyword_score(query, text):
    return len(set(query.lower().split()) & set(text.lower().split()))

DOCS = {"d1": "error code 503 service unavailable", "d2": "the cat sat on the mat",
        "d3": "service mesh routing and load balancing"}

def hybrid(query, alpha=0.5):
    qe = cheap_embed(query)
    ranked = []
    for id, text in DOCS.items():
        score = alpha * cosine(qe, cheap_embed(text)) + (1 - alpha) * keyword_score(query, text)
        ranked.append((round(score, 3), id))
    return sorted(ranked, reverse=True)

print(hybrid("503 service"))